# pybenford — Full Workflow Demo

This notebook walks through every test in the `pybenford` package using US Census county population data (3,195 counties, 2025 estimates).

Population figures are a textbook Benford-conforming dataset: they span several orders of magnitude and arise from natural growth processes. This makes them a good baseline for verifying that the tests work correctly — you should see close conformity across most tests.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pybenford import BenfordAnalysis
from pybenford.visualization import (
    plot_digit_test,
    plot_summation,
    plot_z_scores,
    plot_mantissa_arc,
    plot_ordered_mantissas,
    plot_distortion_factor,
)

%matplotlib inline

## Load Data

In [ ]:
df = pd.read_csv("data/co-est2025-alldata.csv", encoding="latin-1")
print(f"Rows: {len(df):,}")
print(f"Column: POPESTIMATE2025")
print(f"Range: {df['POPESTIMATE2025'].min():,} to {df['POPESTIMATE2025'].max():,}")
print(f"Orders of magnitude: {np.log10(df['POPESTIMATE2025'].max()) - np.log10(df['POPESTIMATE2025'].clip(lower=1).min()):.1f}")

## Create the Analysis Object

Pass any numeric sequence — list, numpy array, or pandas/polars Series. Data cleaning (removing NaN, Inf, zeros, applying sign filters) happens automatically.

In [ ]:
analysis = BenfordAnalysis(df["POPESTIMATE2025"])
print(analysis.profile)

## First Digit Test

The starting point for any Benford analysis. Compares the frequency of leading digits (1-9) against the expected Benford distribution. Nine bins, so results are easy to interpret visually.

In [ ]:
result_d1 = analysis.first_digit()
print(result_d1)

In [ ]:
fig, ax = plot_digit_test(result_d1, show_confidence=True)

## Second Digit Test

Tests the second significant digit (0-9). Less well known than the first digit test but still useful. The expected distribution is flatter than the first digit — digit 0 is most common at ~12%, digit 9 least at ~8.5%.

In [ ]:
result_d2 = analysis.second_digit()
print(result_d2)

In [ ]:
fig, ax = plot_digit_test(result_d2, show_confidence=True)

## First Two Digits Test

The workhorse of forensic Benford analysis (Nigrini Ch. 5). Tests 90 digit combinations (10-99) against expected Benford proportions. With more bins, this test is more sensitive to localized anomalies than the first digit test alone.

The formatted output automatically shows only flagged digits when there are more than 10 bins.

In [ ]:
result_d12 = analysis.first_two_digits()
print(result_d12)

In [ ]:
fig, ax = plot_digit_test(result_d12, show_confidence=True)

## First Three Digits Test

900 bins (100-999). Requires a large dataset to be meaningful — with n=3,195 the average expected count per bin is only ~3.5, which pushes chi-square into unreliable territory. Useful for very large datasets (n > 50,000).

In [ ]:
result_d123 = analysis.first_three_digits()
print(result_d123)

## Last Two Digits Test

Tests the last two digits (00-99) against a uniform distribution (each combination expected at 1%). This is not a Benford test per se — it detects rounding, truncation, and number invention. Fabricated data often has excess 00s and 50s.

In [ ]:
result_last2 = analysis.last_two_digits()
print(result_last2)

## Second Order Test

Sorts the data, computes successive differences, and runs a digit test on those differences (Nigrini Ch. 6). The differences of a Benford-conforming dataset should themselves follow Benford's Law.

This test has very few false positives. If it flags nonconformity, something unusual is happening in the data.

In [ ]:
result_so = analysis.second_order()
print(result_so)

In [ ]:
fig, ax = plot_digit_test(result_so, show_confidence=True)

## Summation Test

Groups values by their first two digits (10-99) and computes the sum of each group as a proportion of the total. Under Benford's Law, each group should contribute approximately 1/90 (~1.11%) of the total sum.

Spikes indicate either a few very large values or clusters of repeated medium-sized values sharing the same leading digits. This is the test that catches individually large outliers.

In [ ]:
result_sum = analysis.summation()
print(result_sum)

In [ ]:
fig, ax = plot_summation(result_sum)

## Distortion Factor

Quantifies systematic over- or understatement (Nigrini Ch. 6). Each value is collapsed to the range [10, 100) and the mean of these collapsed values is compared against the theoretical Benford mean of 39.0865.

A positive distortion factor indicates overstatement (values are systematically larger than expected); negative indicates understatement. The Z-test tells you whether the deviation is statistically significant.

In [ ]:
result_df = analysis.distortion_factor()
print(result_df)

In [ ]:
fig, ax = plot_distortion_factor(result_df)

## Mantissa Arc Test

Tests whether the mantissas (fractional part of log10 values) are uniformly distributed on [0, 1) (Nigrini Ch. 7, Alexander 2009).

Each mantissa is mapped to a point on the unit circle. If the data conforms to Benford's Law, these points should be uniformly distributed around the circle, with the center of gravity near the origin (0, 0). The L-squared statistic measures how far the center of gravity deviates from (0, 0).

In [ ]:
result_ma = analysis.mantissa_arc()
print(result_ma)

In [ ]:
fig, ax = plot_mantissa_arc(result_ma, analysis.clean_data)

In [ ]:
fig, ax = plot_ordered_mantissas(analysis.clean_data)

## Number Duplication

Lists the most frequently repeated values in the dataset (Nigrini Ch. 5). This is the drill-down test — once you've identified suspicious digit patterns via the first-two digits or summation tests, number duplication tells you which specific values are driving those patterns.

In forensic contexts, excessively duplicated values near approval thresholds (e.g., many invoices at $4,999 just below a $5,000 approval limit) are a classic fraud indicator.

In [ ]:
result_dup = analysis.number_duplication(top_n=20)
print(result_dup)

## Working with Results Programmatically

Every result object exposes its data as attributes. This is useful when you need to feed results into downstream analysis or build custom reports.

In [ ]:
result = analysis.first_digit()

print(f"Test:            {result.test_name}")
print(f"Records:         {result.n:,}")
print(f"Alpha:           {result.alpha}")
print(f"MAD:             {result.mad:.6f}")
print(f"MAD Conformity:  {result.mad_conformity}")
print(f"Chi-Square:      {result.chi_square:.4f} (critical: {result.chi_square_critical:.4f})")
print(f"Chi-Square Sig:  {result.chi_square_significant}")
print(f"KS Statistic:    {result.ks_statistic:.4f} (critical: {result.ks_critical:.4f})")
print(f"KS Significant:  {result.ks_significant}")

In [ ]:
# Per-digit breakdown as a DataFrame
digit_df = pd.DataFrame({
    "digit": result.digits,
    "count": result.counts,
    "observed": result.observed,
    "expected": result.expected,
    "z_score": result.z_scores,
    "significant": result.significant_flags,
})
digit_df

## Z-Score Visualization

A dedicated Z-score plot shows which digits deviate most from the expected Benford distribution. The dashed lines mark the critical value at the chosen alpha.

In [ ]:
fig, ax = plot_z_scores(result_d12)

## Using sign_filter for Forensic Workflows

In forensic accounting, income and expense items should be analyzed separately. The `sign_filter` parameter handles this. Here we simulate a mixed dataset to demonstrate.

In [ ]:
# Simulate a mixed income/expense dataset
np.random.seed(42)
mixed_data = np.concatenate([
    np.random.lognormal(mean=8, sigma=1.5, size=3000),   # income (positive)
    -np.random.lognormal(mean=7, sigma=1.2, size=2000),  # expenses (negative)
])

# Analyze positive values only (income)
income_analysis = BenfordAnalysis(mixed_data, sign_filter="positive")
print(f"Income records: {income_analysis.profile}")
print(income_analysis.first_digit())

print("\n")

# Analyze negative values only (expenses)
expense_analysis = BenfordAnalysis(mixed_data, sign_filter="negative")
print(f"Expense records: {expense_analysis.profile}")
print(expense_analysis.first_digit())